In [ ]:
from helpers import RANDOM_STATE, evaluate, load_data, results_table

X, y = load_data()
print("Features:", X.shape)
print("Purchases:", int(y.sum()))

Features: (9247, 17)
Purchases: 1431


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
tree_models = {
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}
baseline_results = [
    evaluate(name, model, X, y, scale=False)
    for name, model in tree_models.items()
]
results_table(baseline_results)

,train_f1,cv_f1,cv_f1_std,cv_precision,cv_recall,cv_accuracy,fit_time_s
model,,,,,,,
Gradient Boosting,0.735,0.661,0.027,0.712,0.617,0.902,3.735
Random Forest,1.000,0.617,0.030,0.758,0.521,0.900,2.144
Decision Tree,1.000,0.563,0.024,0.561,0.565,0.864,0.201


In [ ]:
from sklearn.model_selection import GridSearchCV
from helpers import CV, SCORING, make_model_pipeline
param_grids = {
    "Gradient Boosting": {
        "model__n_estimators": [100, 150],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [2, 3],
    },
    "Random Forest": {
        "model__max_depth": [None, 8, 16],
        "model__min_samples_leaf": [1, 5],
        "model__class_weight": [None, "balanced"],
    },}
tuned_results = []
searches = {}
for name, param_grid in param_grids.items():
    print(f"Tuning {name}...", flush=True)
    search = GridSearchCV(
        estimator=make_model_pipeline(tree_models[name], scale=False),
        param_grid=param_grid,
        scoring=SCORING,
        refit="f1",
        cv=CV,
        n_jobs=-1,
        return_train_score=True,
    )
    search.fit(X, y)
    searches[name] = search
    best = search.best_index_
    scores = search.cv_results_
    tuned_results.append({
        "model": f"{name} (tuned)",
        "train_f1": scores["mean_train_f1"][best],
        "cv_f1": scores["mean_test_f1"][best],
        "cv_f1_std": scores["std_test_f1"][best],
        "cv_precision": scores["mean_test_precision"][best],
        "cv_recall": scores["mean_test_recall"][best],
        "cv_accuracy": scores["mean_test_accuracy"][best],
    })
    print(f"Best parameters: {search.best_params_}")
    print(f"Best CV F1: {search.best_score_:.3f}\n")

Tuning Gradient Boosting...
Best parameters: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 150}
Best CV F1: 0.670

Tuning Random Forest...
Best parameters: {'model__class_weight': 'balanced', 'model__max_depth': None, 'model__min_samples_leaf': 5}
Best CV F1: 0.682



In [ ]:
from pprint import pprint

for name, search in searches.items():
    print(f"{name} best parameters")
    pprint(search.best_params_)
    print()

for row, search in zip(tuned_results, searches.values()):
    row["fit_time_s"] = search.cv_results_["mean_fit_time"][search.best_index_]

results_table(baseline_results + tuned_results)

Gradient Boosting best parameters
{'model__learning_rate': 0.05,
 'model__max_depth': 3,
 'model__n_estimators': 150}

Random Forest best parameters
{'model__class_weight': 'balanced',
 'model__max_depth': None,
 'model__min_samples_leaf': 5}



,train_f1,cv_f1,cv_f1_std,cv_precision,cv_recall,cv_accuracy,fit_time_s
model,,,,,,,
Random Forest (tuned),0.785,0.682,0.008,0.603,0.785,0.887,1.702
Gradient Boosting (tuned),0.720,0.670,0.020,0.726,0.623,0.905,6.055
Gradient Boosting,0.735,0.661,0.027,0.712,0.617,0.902,3.735
Random Forest,1.000,0.617,0.030,0.758,0.521,0.900,2.144
Decision Tree,1.000,0.563,0.024,0.561,0.565,0.864,0.201


## Tree Model Results

The gradient boosting algorithm achieved the highest baseline F1 score (0.661) for the purchase category, followed by Random Forest (0.617) and Decision Tree (0.563).

Both Random Forest and Decision Tree had a mean training F1 score of 1.000, but their lower validation scores indicated overfitting. After grid search, Random Forest achieved the highest mean validation F1 score (0.682 ± 0.008) among all tree models. Its optimal parameter settings were: `class_weight="balanced"`, `max_depth=None`, and `min_samples_leaf=5`. Compared to the baseline, recall improved from 0.521 to 0.785, while precision decreased from 0.758 to 0.603. The adjusted model captured more purchasing behavior but also incorrectly identified more non-purchasing behaviors as purchasing behaviors.

After grid search, Gradient Boosting’s mean validation F1 for the purchase class increased from 0.661 to 0.670 ± 0.020. Its best settings were `learning_rate=0.05`, `max_depth=3`, and `n_estimators=150`. Tuned Gradient Boosting had higher validation precision than tuned Random Forest (0.726 versus 0.603), while tuned Random Forest had higher validation recall (0.785 versus 0.623).

Grid search used the same five stratified cross-validation folds to select hyperparameters and report the tuned scores. All model development used the training data.